In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, os, subprocess, sys
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXACT='70b675efa33ce82d02129bffb72dd8429b023e8a'
R0_UNIT_ID='geometry-v6-r0-dev-0001'
PROMPT='A watchmaker sorting steel springs beneath a magnifying lamp'
SEED=2026082400
HEIGHT=512
WIDTH=512
REPO=Path('/content/cegwm-geometry-v6-r0'); DRIVE_RUNS=Path('/content/drive/MyDrive/CEG-WM/Geometry-V6/R0')
if REPO.exists(): raise FileExistsError('fresh runtime required')
subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True); subprocess.run(['git','checkout','--detach',APPROVED_EXACT],cwd=REPO,check=True)
def git(*args): return subprocess.run(['git',*args],cwd=REPO,check=True,capture_output=True,text=True).stdout.strip()
assert git('rev-parse','HEAD')==APPROVED_EXACT and git('branch','--show-current')=='' and git('status','--porcelain')==''
RUN_ROOT=DRIVE_RUNS/f'{APPROVED_EXACT}-{R0_UNIT_ID}-{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}'
if RUN_ROOT.exists(): raise FileExistsError('create-only run exists')
RUN_ROOT.mkdir(parents=True,exist_ok=False)


In [ ]:
from google.colab import userdata
assert __import__('torch').cuda.is_available(), 'GPU required; no R0 record on CPU'
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
content_key=userdata.get('CEG_WM_ROOT_KEY'); hf_token=userdata.get('HF_TOKEN')
assert all(isinstance(value,str) and value.strip() for value in (content_key,hf_token))
markers=('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'); runner_env={key:value for key,value in os.environ.items() if not any(marker in key.upper() for marker in markers)}
runner_env.update({'CEG_WM_ROOT_KEY':content_key,'HF_TOKEN':hf_token})
command=[sys.executable,'-m','experiments.geometry_v6_r0_engine','--run-diagnostic','--repo-root',str(REPO),'--expected-exact',APPROVED_EXACT,'--prompt',PROMPT,'--seed',str(SEED),'--height',str(HEIGHT),'--width',str(WIDTH),'--output-json',str(RUN_ROOT/'r0.json')]
try: completed=subprocess.run(command,cwd=REPO,env=runner_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,check=False)
finally: content_key=hf_token=''; runner_env.clear()
summary=completed.stdout.strip(); print(summary)
if completed.returncode!=0: raise RuntimeError('Geometry-V6 R0 stopped; retained create-only path: '+str(RUN_ROOT))
artifact=RUN_ROOT/'r0.json'; print({'path':str(artifact),'sha256':hashlib.sha256(artifact.read_bytes()).hexdigest(),'exact':APPROVED_EXACT})
